# ⚠️ Required First Step — Enable GPU

1. Click **Runtime** in the menu above
2. Click **Change runtime type**
3. Set **Hardware accelerator** to **T4 GPU** (free) or **A100**
4. Click **Save**

Then click **Runtime → Run all** (Ctrl+F9 / Cmd+F9). Takes ~3 minutes.

---

# TunedAI Labs — Causal Reasoning Demo

Compares two versions of the same model on ten causal reasoning questions:

| Model | What it is |
|---|---|
| Base Qwen 2.5-7B | Unmodified open-source model |
| **TunedAI Labs Causal Model** | Same model, fine-tuned on 10,112 causal reasoning questions |

Questions use fictional variable names — the models cannot recall the answer from pretraining.
Scored on correct YES/NO only.

**Benchmark:** Fine-tuned model scores **96.96%** on CLadder. Base Qwen scores ~62%.

---

## Step 1 — Install packages

In [ ]:
import subprocess, sys

pkgs = ['transformers', 'peft', 'accelerate', 'bitsandbytes', 'openai', 'anthropic']
print(f"Installing {len(pkgs)} packages...")
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs,
                   capture_output=True, text=True)
if r.returncode != 0:
    print("Warning:", r.stderr[-200:])

print("Verifying:")
ok = True
for pkg in pkgs:
    try:
        __import__(pkg if pkg != 'bitsandbytes' else 'bitsandbytes')
        print(f"  {pkg} ✓")
    except ImportError:
        print(f"  {pkg} ✗")
        ok = False
if ok:
    print("\n✓ All packages ready.")
else:
    print("\n⚠ Some packages failed — try running this cell again.")

---
## Step 2 — API keys (optional)

Leave blank to skip GPT-4o and Claude columns — the local model comparison still runs.

In [ ]:
OPENAI_API_KEY    = ""   # paste OpenAI key here (optional)
ANTHROPIC_API_KEY = ""   # paste Anthropic key here (optional)

print("OpenAI key:",    "set" if OPENAI_API_KEY    else "not set (GPT-4o column will be skipped)")
print("Anthropic key:", "set" if ANTHROPIC_API_KEY else "not set (Claude column will be skipped)")

---
## Step 3 — Load models

Downloads Qwen 2.5-7B-Instruct + TunedAI Labs LoRA adapter from HuggingFace.
Takes ~90 seconds on T4.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import openai, anthropic

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Go to Runtime → Change runtime type → set T4 GPU.")
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {torch.cuda.get_device_name(0)}  |  VRAM: {vram_gb:.0f} GB")

BASE_MODEL   = "Qwen/Qwen2.5-7B-Instruct"
ADAPTER_REPO = "tunedailabs/causal-reasoning-qwen-7b"

print("Loading tokenizer...", end=" ", flush=True)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
print("✓")

if vram_gb >= 20:
    print("Loading base model float16 (~90 sec)...", end=" ", flush=True)
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, torch_dtype=torch.float16, device_map="cuda:0", trust_remote_code=True)
else:
    print("Loading base model 8-bit (~90 sec)...", end=" ", flush=True)
    bnb_config = BitsAndBytesConfig(load_in_8bit=True)
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, quantization_config=bnb_config, device_map="auto", trust_remote_code=True)
print("✓")

print("Loading TunedAI Labs adapter...", end=" ", flush=True)
tuned_model = PeftModel.from_pretrained(base_model, ADAPTER_REPO)
tuned_model.eval()
print("✓")

oai_client = openai.OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None
ant_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY) if ANTHROPIC_API_KEY else None

print("\n✓ Models ready.")

---
## Step 4 — Verify adapter integrity

Checks SHA256 of the downloaded adapter against the known-good v2_e1 hash (96.96% CLadder).

In [ ]:
import hashlib, os, glob

# SHA256 of the correct v2_e1 adapter (96.96% CLadder, 9805/10112 correct)
KNOWN_SHA256 = "e0967ac8460870eab2f114379c4d4e9640934f944e294e115eec3f6bc67eba13"

# Find the cached adapter file (HuggingFace downloads to ~/.cache/huggingface)
cache_pattern = os.path.expanduser(
    "~/.cache/huggingface/hub/models--tunedailabs--causal-reasoning-qwen-7b/**/*.safetensors")
candidates = [f for f in glob.glob(cache_pattern, recursive=True)
              if not f.endswith(".incomplete")]

if not candidates:
    print("⚠ Adapter not found in cache — run Step 3 first.")
else:
    adapter_file = candidates[0]
    print(f"Verifying: {os.path.basename(adapter_file)} ({os.path.getsize(adapter_file)/1e6:.0f} MB)")
    print("Computing SHA256...", end=" ", flush=True)

    sha = hashlib.sha256()
    with open(adapter_file, "rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            sha.update(chunk)
    computed = sha.hexdigest()

    if computed == KNOWN_SHA256:
        print("✓")
        print(f"\n✓ SHA256 MATCH — confirmed v2_e1 adapter (96.96% CLadder)")
        print(f"  {computed}")
    else:
        print("✗")
        print(f"\n✗ SHA256 MISMATCH — wrong adapter loaded")
        print(f"  Expected: {KNOWN_SHA256}")
        print(f"  Got:      {computed}")
        print("\nFix: Runtime → Disconnect and delete runtime, then start over.")
        raise RuntimeError("Wrong adapter. Results would be invalid.")

---
## Step 5 — Run comparison

Ten causal reasoning questions with fictional variable names.
Base Qwen scores ~62% on CLadder. TunedAI scores 96.96%.
Takes ~5 minutes.

In [ ]:
import json, os, torch

SYSTEM = "You are a causal reasoning expert. Answer YES or NO only."
RESULTS_FILE = "/content/tunedai_results.json"

QUESTIONS = [
    # Rung 1 — negative association
    ("The overall probability of disease Zorath is 15%. "
     "For people who consume mineral Velyx, the probability of Zorath is 3%. "
     "For people who do not consume Velyx, the probability of Zorath is 22%. "
     "Is consuming Velyx negatively associated with disease Zorath?",
     "yes"),
    # Rung 1 — correlation direction
    ("The overall probability of crop failure in Ferdox province is 45%. "
     "For farms using blue fertilizer, crop failure rate is 30%. "
     "For farms not using blue fertilizer, crop failure rate is 55%. "
     "Is blue fertilizer positively correlated with crop failure?",
     "no"),
    # Rung 1 — common cause
    ("In the town of Delmor, swimming pool registrations and ice cream vendor licenses "
     "both increased together over 10 years, with correlation r = 0.96. "
     "Does opening more swimming pools cause ice cream vendors to open?",
     "no"),
    # Rung 2 — Simpson's paradox with fictional drugs
    ("Drug Trelix cures 90% of mild cases and 50% of severe cases. "
     "Drug Sorrin cures 80% of mild cases and 40% of severe cases. "
     "Sorrin is given more often to mild patients, giving it an overall cure rate of 78% vs Trelix's 65%. "
     "Does Sorrin cause better patient outcomes than Trelix?",
     "no"),
    # Rung 2 — confounding
    ("In the village of Kessler, people who carry umbrellas have higher rates of wet clothing "
     "than people who do not carry umbrellas. Rain causes both umbrella carrying and wet clothing. "
     "If we mandate that everyone carry umbrellas, will rates of wet clothing decrease?",
     "no"),
    # Rung 2 — valid physical intervention
    ("In factory Dormund, workers who choose to wear protective goggles have a 2% eye injury rate. "
     "Workers who do not wear goggles have a 25% rate. "
     "Goggles physically block debris from reaching the eyes. "
     "If we mandate goggles for all workers, will eye injury rates decrease?",
     "yes"),
    # Rung 2 — causal chain
    ("Mineral Velyx is known to cause a reduction in protein Zanthor. "
     "High levels of protein Zanthor are known to cause disease Gruuk. "
     "Does consuming Velyx cause a reduction in Gruuk risk?",
     "yes"),
    # Rung 3 — counterfactual with known causal mechanism
    ("Mira took medication Pellax and recovered from infection Gruuk within 4 days. "
     "Pellax is known to cause recovery from Gruuk by neutralizing it. "
     "If Mira had not taken Pellax, would she have recovered from Gruuk within 4 days?",
     "no"),
    # Rung 3 — counterfactual with alternate cause
    ("City Velmor installed decorative blue streetlights in 2022 and crime fell 40%. "
     "Research confirmed blue light itself does not reduce crime — "
     "the increased police patrols that accompanied installation were the actual cause. "
     "If the blue streetlights had never been installed but police patrols remained the same, "
     "would crime have still fallen?",
     "yes"),
    # Rung 1 — survivorship bias
    ("Analysts studied 300 successful companies in Krenthar and found they all expanded "
     "aggressively in their first year. Companies that expanded aggressively and failed "
     "were not included — they no longer exist. "
     "Does the evidence support the conclusion that aggressive expansion causes company success?",
     "no"),
]

def _detect_yn(text):
    t = text.strip().lower()
    words = t.split()
    first = words[0].rstrip(".,!:") if words else ""
    if first == "yes": return "yes"
    if first == "no":  return "no"
    if t.startswith("yes"): return "yes"
    if t.startswith("no"):  return "no"
    snippet = t[:60]
    if "yes" in snippet and "no" not in snippet: return "yes"
    if "no" in snippet and "yes" not in snippet: return "no"
    return "?"

def ask_local(question, use_adapter=True):
    if use_adapter: tuned_model.enable_adapter_layers()
    else:           tuned_model.disable_adapter_layers()
    messages = [{"role":"system","content":SYSTEM},{"role":"user","content":question}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = tuned_model.generate(**inputs, max_new_tokens=80, do_sample=False)
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

def ask_gpt4(q):
    if not oai_client: return "[No OpenAI key]"
    r = oai_client.chat.completions.create(model="gpt-4o",
        messages=[{"role":"system","content":SYSTEM},{"role":"user","content":q}], max_tokens=80)
    return r.choices[0].message.content.strip()

def ask_claude(q):
    if not ant_client: return "[No Anthropic key]"
    r = ant_client.messages.create(model="claude-3-5-sonnet-20241022", max_tokens=80,
        system=SYSTEM, messages=[{"role":"user","content":q}])
    return r.content[0].text.strip()

# Load saved results (safe to interrupt and resume)
results = {}
if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE) as f:
        results = json.load(f)
    print(f"Resuming — {len(results)} of {len(QUESTIONS)} done.")

SEP = "="*70; DIV = "-"*70

for i, (question, expected) in enumerate(QUESTIONS):
    key = str(i)
    if key in results:
        r = results[key]
        b = "✓" if r["base_yn"] == expected else "✗"
        t = "✓" if r["tuned_yn"] == expected else "✗"
        print(f"[Q{i+1} done — Base {b} · TunedAI {t}]")
        continue

    print(f"\n{SEP}")
    print(f"Q{i+1}/{len(QUESTIONS)}  |  Expected: {expected.upper()}")
    print(SEP)
    print(f"{question}\n")

    base_ans  = ask_local(question, use_adapter=False)
    gpt_ans   = ask_gpt4(question)
    cld_ans   = ask_claude(question)
    tuned_ans = ask_local(question, use_adapter=True)

    b_yn = _detect_yn(base_ans)
    t_yn = _detect_yn(tuned_ans)
    b_ok = b_yn == expected
    t_ok = t_yn == expected

    for label, ans in [("BASE QWEN 2.5-7B", base_ans), ("GPT-4o", gpt_ans),
                       ("CLAUDE 3.5 SONNET", cld_ans), ("TUNEDAI LABS ★", tuned_ans)]:
        print(DIV); print(f"[ {label} ]"); print(DIV); print(ans); print()

    star = " ★" if t_ok and not b_ok else (" ↓" if b_ok and not t_ok else "")
    print(f"  Base: {b_yn.upper()} ({'✓' if b_ok else '✗'})  |  TunedAI: {t_yn.upper()} ({'✓' if t_ok else '✗'}){star}")

    results[key] = {"base_ans":base_ans,"tuned_ans":tuned_ans,
                    "base_yn":b_yn,"tuned_yn":t_yn,"expected":expected}
    with open(RESULTS_FILE,"w") as f: json.dump(results,f,indent=2)

print(f"\n{SEP}\nAll questions complete.")

---
## Results

In [ ]:
import json, os
RESULTS_FILE = "/content/tunedai_results.json"
QUESTIONS_N  = 10

if not os.path.exists(RESULTS_FILE):
    print("No results yet — run the cell above first.")
else:
    with open(RESULTS_FILE) as f: res = json.load(f)
    n = len(res)

    b_correct = sum(1 for r in res.values() if r["base_yn"]  == r["expected"])
    t_correct = sum(1 for r in res.values() if r["tuned_yn"] == r["expected"])

    SEP = "="*70
    print(SEP)
    print("FINAL RESULTS — TunedAI Labs Causal Reasoning Demo")
    print(f"{n} causal YES/NO questions · scored on correct answer only")
    print(SEP)
    print(f"  Base Qwen 2.5-7B (untuned) : {b_correct:2d}/{n} = {100*b_correct/n:.1f}%")
    print(f"  TunedAI Labs Causal Model ★: {t_correct:2d}/{n} = {100*t_correct/n:.1f}%")
    diff = 100*t_correct/n - 100*b_correct/n
    print(f"  Improvement                : {diff:+.1f} percentage points")
    print(SEP)
    print(f"\n{'Q':<5} {'Expected':>10} {'Base':>8} {'TunedAI':>10}")
    print(f"{'----':<5} {'--------':>10} {'----':>8} {'-------':>10}")
    for i in range(n):
        k = str(i)
        if k not in res: continue
        r = res[k]
        b = "✓" if r["base_yn"]  == r["expected"] else "✗"
        t = "✓" if r["tuned_yn"] == r["expected"] else "✗"
        flag = " ★" if t=="✓" and b=="✗" else (" ↓" if b=="✓" and t=="✗" else "")
        print(f"Q{i+1:<4} {r['expected'].upper():>10} {r['base_yn'].upper():>5} {b:>3}  {r['tuned_yn'].upper():>5} {t:>3}{flag}")

---
## Try Your Own Question

The model was trained on short structured causal questions. Use this template:

**Template:**
> In [setting], [Group A] [did X]. [Group B] [did not do X].
> [Group A result]. [Group B result].
> Does [X] cause [outcome]?

Replace the example below and run the cell.

In [ ]:
MY_QUESTION = """
Factory workers were split into two groups. Group A received ergonomic chairs. Group B kept standard chairs.
After 6 months, Group A reported 40% less back pain. Group B reported no change.
Does using an ergonomic chair cause a reduction in back pain?
"""

base_ans  = ask_local(MY_QUESTION.strip(), use_adapter=False)
tuned_ans = ask_local(MY_QUESTION.strip(), use_adapter=True)

print(f"Question: {MY_QUESTION.strip()}\n")
print(f"[ BASE QWEN 2.5-7B ]\n{base_ans}\n")
print(f"[ TUNEDAI LABS ★ ]\n{tuned_ans}")

---
## Share Your Results

Open a [GitHub Issue](https://github.com/tunedailabs/tunedailabs/issues/new) and paste what you saw.

**TunedAI Labs** — We fine-tune open-source LLMs for real-world reasoning tasks.
[tunedailabs.com](https://tunedailabs.com)